In [1]:
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import json

In [ ]:
dataset_a = [
    {"username": "alice_smith", "email": "alice@example.com", "age": 28, "signup_date": "2023-10-01"},
    {"username": "bob_jones", "email": "bob.jones@example.com", "age": 34, "signup_date": "2023-10-02"},
    {"username": "charlie_davis", "email": "charlie@example.com", "age": 22, "signup_date": "2023-10-03"},
    {"username": "diana_prince", "email": "diana@example.com", "age": 30, "signup_date": "2023-10-04"},
    {"username": "evan_wright", "email": "evan@example.com", "age": 45, "signup_date": "2023-10-05"},
    {"username": "frank_castle", "email": "frankcastle.com", "age": 38, "signup_date": "2023-10-06"},
    {"username": "grace_hopper", "email": "grace@example.com", "age": "22", "signup_date": "2023-10-07"},
    {"username": "", "email": "unknown.user@example.com", "age": 19, "signup_date": "2023-10-08"}
]

data rep: mix of user input and sys generated because signup date is not something which user inputs. All the other datas are user input.

Quality issue: issue is likely might be wrong datas about users, which is users might be lying about their personal infos like age or something

validations: checking type of ages

In [9]:
dataset_b = [
    {"timestamp": "2023-10-10T08:00:01Z", "service": "AuthService", "level": "INFO", "message": "User login successful."},
    {"timestamp": "2023-10-10T08:05:23Z", "service": "PaymentAPI", "level": "INFO", "message": "Payment processing initiated."},
    {"timestamp": "2023-10-10T08:05:45Z", "service": "PaymentAPI", "level": "WARNING", "message": "High latency detected from payment gateway."},
    {"timestamp": "2023-10-10T08:06:12Z", "service": "DatabaseService", "level": "INFO", "message": "Database backup completed successfully."},
    {"timestamp": "2023-10-10T08:15:00Z", "service": "AuthService", "level": "WARNING", "message": "Multiple failed login attempts for user ID 402."},
    {"timestamp": "2023-10-10T08:17:33Z", "service": "PaymentAPI", "level": "ERROR", "message": "Payment gateway timeout. Transaction failed."},
    {"timestamp": "2023-10-10T08:20:05Z", "service": "DatabaseService", "level": "ERROR", "message": "Connection pool exhausted."},
    {"timestamp": "2023-10-10T08:22:10Z", "service": "AuthService", "level": "INFO", "message": "Password reset email sent to user ID 402."},
    {"timestamp": "2023-10-10T08:30:00Z", "service": "DatabaseService", "level": "INFO", "message": "Connection pool restored."},
    {"timestamp": "2023-10-10T08:45:11Z", "service": "PaymentAPI", "level": "INFO", "message": "Payment processing retry successful."}
]

data rep: system generated

quality issue:error and warning messages are too short to interpret and understand the problem, timestamps does not include milli or microseconds, which is crucial for large databases which gets hundreds of loggings each second. and there is no way to trace users from message and services, if payment is failed or illegally large for one users, we cant find who is this user

validation: add micro and milli seconds to timestamps

In [8]:
dataset_c = [
    {"sku": "LAP-X1-16GB", "product_name": "ProBook X1 Laptop", "quantity_in_stock": 145, "warehouse_location": "Zone A - NY", "last_updated": "2023-10-09"},
    {"sku": "MON-27-4K", "product_name": "UltraView 27-inch 4K Monitor", "quantity_in_stock": 32, "warehouse_location": "Zone B - CA", "last_updated": "2023-10-08"},
    {"sku": "KBD-MECH-RGB", "product_name": "ClickMaster Mechanical Keyboard", "quantity_in_stock": 0, "warehouse_location": "Zone A - NY", "last_updated": "2023-10-05"},
    {"sku": "MSE-WIR-PRO", "product_name": "GlidePro Wireless Mouse", "quantity_in_stock": 210, "warehouse_location": "Zone C - TX", "last_updated": "2023-10-09"},
    {"sku": "CBL-USB-C-2M", "product_name": "Braided USB-C Cable (2m)", "quantity_in_stock": 850, "warehouse_location": "Zone B - CA", "last_updated": "2023-10-01"},
    {"sku": "DOCK-STN-UNV", "product_name": "Universal USB-C Docking Station", "quantity_in_stock": 15, "warehouse_location": "Zone C - TX", "last_updated": "2023-10-10"}
]

data rep: internal database of a company

quality issue: last update is not enough precise, if a customer would buy product in a same day, stock will not be updated till tomorrow.

validation: add hours to the last update, split location to two parts, zone and state, check if stocks are numeric type

In [10]:
def validate_registrations(entries):
    valid_entries = []
    invalid_entries = []

    for entry in entries:
        is_valid = True
        
        # 1. Check for non-empty username
        username = entry.get("username", "")
        if not username or str(username).strip() == "":
            is_valid = False
            
        # 2. Check for '@' in email
        email = entry.get("email", "")
        if "@" not in str(email):
            is_valid = False
            
        # 3. Check if age can be converted to a positive integer
        try:
            age = int(entry.get("age"))
            if age <= 0:
                is_valid = False
        except (ValueError, TypeError):
            # Catches strings like "eighty-five" or None types
            is_valid = False
            
        # Route the entry to the appropriate list
        if is_valid:
            valid_entries.append(entry)
        else:
            invalid_entries.append(entry)
            
    return valid_entries, invalid_entries

# Assuming dataset_a from Step 1 is loaded in memory
valid, invalid = validate_registrations(dataset_a)

print(f"Total entries processed: {len(dataset_a)}")
print(f"Valid entries count: {len(valid)}")
print(f"Invalid entries count: {len(invalid)}")

Total entries processed: 8
Valid entries count: 5
Invalid entries count: 3


In [19]:
np.random.seed(42)

rows = 500
rides=pd.DataFrame({
    'ride_id': range(1,rows+1),
    'driver_id': np.random.randint(1000,1051,size=rows),
    'distance_km': np.random.uniform(1.0,30.0,size=rows).round(2),
    'fare_usd': np.random.uniform(5.0,75.0, size=rows).round(2),
    'ride_type': np.random.choice(['standard','premium','pool'],size=rows),
    'city': np.random.choice(['Berlin','Seoul','Nairobi','Toronto','Lima'],size=rows),
    'timestamps': pd.date_range(start='2025-01-01',periods=rows,freq='min')
})
rides = rides.set_index('ride_id')

rides.head()


,driver_id,distance_km,fare_usd,ride_type,city,timestamps
ride_id,,,,,,
1,1038,3.66,45.43,standard,Berlin,2025-01-01 00:00:00
2,1028,3.73,16.59,pool,Seoul,2025-01-01 00:01:00
3,1014,10.03,7.37,pool,Seoul,2025-01-01 00:02:00
4,1042,29.41,26.80,premium,Lima,2025-01-01 00:03:00
5,1007,6.08,59.63,pool,Nairobi,2025-01-01 00:04:00


In [22]:
rides.to_csv('rides_csv',index=True)

In [31]:
rides.reset_index().to_json('rides.jsonl',orient="records", lines=True,date_format='iso')

In [32]:
rides.to_parquet('rides.parquet')

In [34]:
import os

json_size = os.path.getsize("rides.jsonl")/1024
pq_size = os.path.getsize("rides.parquet")/1024
csv_size = os.path.getsize("rides_csv")/1024

print("json size(in KBs): ",json_size)
print("pq size(in KBs): ",pq_size)
print("csv size(in KBs): ",csv_size)

json size(in KBs):  71.0
pq size(in KBs):  17.5703125
csv size(in KBs):  27.119140625


In [36]:
csv_df = pd.read_csv('rides_csv',index_col=0)
csv_df

,driver_id,distance_km,fare_usd,ride_type,city,timestamps
ride_id,,,,,,
1,1038,3.66,45.43,standard,Berlin,2025-01-01 00:00:00
2,1028,3.73,16.59,pool,Seoul,2025-01-01 00:01:00
3,1014,10.03,7.37,pool,Seoul,2025-01-01 00:02:00
4,1042,29.41,26.80,premium,Lima,2025-01-01 00:03:00
5,1007,6.08,59.63,pool,Nairobi,2025-01-01 00:04:00
...,...,...,...,...,...,...
496,1050,27.64,8.06,pool,Nairobi,2025-01-01 08:15:00
497,1036,18.02,16.32,premium,Nairobi,2025-01-01 08:16:00
498,1021,1.95,6.46,standard,Berlin,2025-01-01 08:17:00


In [48]:
csv_df.shape

(500, 6)

In [44]:
json_df = pd.read_json('rides.jsonl',lines=True).set_index('ride_id')
json_df

,driver_id,distance_km,fare_usd,ride_type,city,timestamps
ride_id,,,,,,
1,1038,3.66,45.43,standard,Berlin,2025-01-01 00:00:00
2,1028,3.73,16.59,pool,Seoul,2025-01-01 00:01:00
3,1014,10.03,7.37,pool,Seoul,2025-01-01 00:02:00
4,1042,29.41,26.80,premium,Lima,2025-01-01 00:03:00
5,1007,6.08,59.63,pool,Nairobi,2025-01-01 00:04:00
...,...,...,...,...,...,...
496,1050,27.64,8.06,pool,Nairobi,2025-01-01 08:15:00
497,1036,18.02,16.32,premium,Nairobi,2025-01-01 08:16:00
498,1021,1.95,6.46,standard,Berlin,2025-01-01 08:17:00


In [47]:
json_df.shape

(500, 6)

In [45]:
parquet_df = pd.read_parquet('rides.parquet')
parquet_df

,driver_id,distance_km,fare_usd,ride_type,city,timestamps
ride_id,,,,,,
1,1038,3.66,45.43,standard,Berlin,2025-01-01 00:00:00
2,1028,3.73,16.59,pool,Seoul,2025-01-01 00:01:00
3,1014,10.03,7.37,pool,Seoul,2025-01-01 00:02:00
4,1042,29.41,26.80,premium,Lima,2025-01-01 00:03:00
5,1007,6.08,59.63,pool,Nairobi,2025-01-01 00:04:00
...,...,...,...,...,...,...
496,1050,27.64,8.06,pool,Nairobi,2025-01-01 08:15:00
497,1036,18.02,16.32,premium,Nairobi,2025-01-01 08:16:00
498,1021,1.95,6.46,standard,Berlin,2025-01-01 08:17:00


In [49]:
parquet_df.shape

(500, 6)

As a matter of readability, csv format is best for me, and as a matter of memory parquet is best, each of them preserved data nicely I guess